In [ ]:
import pandas as pd
Data=pd.read_csv("/content/CTrain.csv")
Data.head()

In [ ]:
#@title INFORMATION ABOUT DATASET
Data.info()

In [ ]:
#@title UNIQUE VALUES
#we can understand which columns are categoricals and which columns are not
Data.apply(lambda x: x.nunique())

In [ ]:
#@title CORRELATION
#our target is SYMPTOMS
Data.corr().style.background_gradient(cmap='coolwarm')

In [ ]:
#@title HISTOGRAM
import seaborn as sns
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
matplotlib.style.use('ggplot')
plt.figure(figsize=(18,20))
for i,col in enumerate(Data.columns,1):
    plt.subplot(5,6,i)
    sns.histplot(Data[col],kde=True)

In [ ]:
from sklearn .preprocessing import LabelEncoder
encode=LabelEncoder()

In [ ]:
Data.loc[:,Data.columns[-1]]=encode.fit_transform(Data.loc[:,Data.columns[-1]])

In [ ]:
encode.classes_

In [ ]:
Data.head()

In [ ]:
from sklearn.model_selection import train_test_split
X_train , X_test = train_test_split(Data,test_size = 0.3)

In [ ]:
df = X_train

In [ ]:
df.head()


In [ ]:
df[df.columns[-1]].value_counts()

In [ ]:
df.corr().style.background_gradient(cmap='coolwarm')

In [ ]:
from sklearn import model_selection
from sklearn.preprocessing import MinMaxScaler,StandardScaler,RobustScaler

In [ ]:
df=df.sample(frac=1).reset_index(drop=True)

In [ ]:
df.columns=df.columns.astype("str")

In [ ]:
col_name=df.columns.to_list()

In [ ]:
col_name

In [ ]:
target_col=col_name.pop(-1)

In [ ]:
col_name.pop(0)

In [ ]:
target_col

In [ ]:
col_name

In [ ]:
y=df["SYMPTOMS"].values
y

In [ ]:
from imblearn.combine import SMOTETomek
from collections import Counter

In [ ]:
df_train=df.drop(df.columns[-1],axis=1)

In [ ]:
df_train_val=df_train.values

In [ ]:
smk=SMOTETomek(random_state=42)
X_res,y_res=smk.fit_resample(df_train_val,y)

In [ ]:
X_res.shape,y_res.shape

In [ ]:
print(f"Orignal Dataset Shape {Counter(y)}")
print(f"Resampled Dataset Shape {Counter(y_res)}")

In [ ]:
df_new=pd.concat([pd.DataFrame(X_res),pd.DataFrame(y_res)],axis=1)

In [ ]:
df_new.head()

In [ ]:
col_new=df_train.columns
col_new=col_new.to_list()
col_new.append(target_col)
df_new.columns=col_new
df_new.head()

In [ ]:
X_train.shape

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix,classification_report,accuracy_score,roc_auc_score
import warnings
warnings.filterwarnings('ignore')
acc_log=[]

In [ ]:
 ## Building a Classification model using 
 kf=model_selection.StratifiedKFold(n_splits=5)
 for fold , (trn_,val_) in enumerate(kf.split(X=df_new,y=y_res)):
    
     X_train=df_new.loc[trn_,col_name]
     y_train=df_new.loc[trn_,target_col]
    
     X_valid=df_new.loc[val_,col_name]
     y_valid=df_new.loc[val_,target_col]
    
     ro_scaler=StandardScaler()
     X_train=ro_scaler.fit_transform(X_train)
     X_valid=ro_scaler.transform(X_valid)
    
     clf=LogisticRegression()
     clf.fit(X_train,y_train)
     y_pred=clf.predict(X_valid)
     print(f"The fold is : {fold} : ")
     print(classification_report(y_valid,y_pred))
     #acc=roc_auc_score(y_valid,y_pred,multi_class="ovo")
     #acc_log.append(acc)
     #print(f"The accuracy for {fold} : {acc}")
